# ARK ETF HHI Concentration Analysis

Portfolio concentration analysis using Herfindahl-Hirschman Index (HHI)

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Configuration

In [ ]:
etf_list = ['ARKK', 'ARKW', 'ARKQ', 'ARKF', 'ARKG', 'ARKX']
start_date = '2025-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')

# Check input data range
sample = pd.read_excel(f'../input/{etf_list[0]}_Transformed_Data.xlsx', sheet_name='Sheet1')
data_start = pd.to_datetime(sample['Date']).min().strftime('%Y-%m-%d')
data_end = pd.to_datetime(sample['Date']).max().strftime('%Y-%m-%d')

print(f"Data range: {data_start} to {data_end}")
print(f"Selected: {start_date} to {end_date}\n")

## Step 1: Load All ETF Data

In [ ]:
etf_data_dict = {}

for etf_name in etf_list:
    print(f"Loading {etf_name} data...")
    file_path = f'../input/{etf_name}_Transformed_Data.xlsx'
    etf_data = pd.read_excel(file_path, sheet_name='Sheet1')
    etf_data['Date'] = pd.to_datetime(etf_data['Date'])
    
    if etf_data['Weight'].max() > 1:
        etf_data['Weight'] = etf_data['Weight'] / 100
    
    cash_funds = ['XX', 'MVRXX', 'DGCXX', 'FEDXX']
    etf_data = etf_data[~etf_data['Ticker'].isin(cash_funds)]
    etf_data = etf_data.sort_values('Date')
    
    etf_data = etf_data[(etf_data['Date'] >= pd.to_datetime(start_date)) & 
                        (etf_data['Date'] <= pd.to_datetime(end_date))]
    
    etf_data_dict[etf_name] = etf_data
    print(f"  ✅ Loaded {len(etf_data)} records, Date range: {etf_data['Date'].min().date()} to {etf_data['Date'].max().date()}")

print(f"\n✅ All {len(etf_data_dict)} ETFs loaded!")

## Step 2: Calculate HHI

In [ ]:
hhi_dict = {}

for etf_name, etf_data in etf_data_dict.items():
    print(f"Calculating HHI for {etf_name}...")
    unique_dates = sorted(etf_data['Date'].unique())
    hhi_results = []
    
    for current_date in unique_dates:
        daily_holdings = etf_data[etf_data['Date'] == current_date]
        weights = daily_holdings['Weight'].values
        hhi = np.sum(weights ** 2)
        enh = 1 / hhi
        
        hhi_results.append({
            'Date': current_date.strftime('%m/%d/%Y'),
            'HHI': hhi,
            'ENH': enh,
            'Holdings_Count': len(daily_holdings)
        })
    
    hhi_df = pd.DataFrame(hhi_results)
    hhi_dict[etf_name] = hhi_df
    print(f"  ✅ {len(hhi_df)} days, Mean HHI: {hhi_df['HHI'].mean():.4f}, Mean ENH: {hhi_df['ENH'].mean():.2f}")

print(f"\n✅ HHI calculated for all {len(hhi_dict)} ETFs!")

## Step 3: Calculate P&L and Top Contributors

In [ ]:
analysis_dict = {}
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

for etf_name, etf_data in etf_data_dict.items():
    print(f"Calculating P&L for {etf_name}...")
    unique_dates = sorted(etf_data['Date'].unique())
    contributor_results = []
    
    for i, current_date in enumerate(unique_dates):
        if i == 0:
            contributor_results.append({
                'Date': current_date.strftime('%m/%d/%Y'),
                'Top_50pct_Profit_Count': 0,
                'Top_50pct_Profit_Tickers': '',
                'Top_50pct_Loss_Count': 0,
                'Top_50pct_Loss_Tickers': ''
            })
        else:
            prev_holdings = etf_data[etf_data['Date'] == unique_dates[i-1]]
            daily_holdings = etf_data[etf_data['Date'] == current_date]
            
            all_tickers = set(prev_holdings['Ticker'].unique()) | set(daily_holdings['Ticker'].unique())
            pnl_data = {}
            
            for ticker in all_tickers:
                start_rows = prev_holdings[prev_holdings['Ticker'] == ticker]
                end_rows = daily_holdings[daily_holdings['Ticker'] == ticker]
                
                day0_pos = start_rows.iloc[0]['Position'] if len(start_rows) > 0 else 0
                day0_price = start_rows.iloc[0]['Stock_Price'] if len(start_rows) > 0 else 0
                day0_mv = day0_pos * day0_price
                
                day1_pos = end_rows.iloc[0]['Position'] if len(end_rows) > 0 else 0
                day1_price = end_rows.iloc[0]['Stock_Price'] if len(end_rows) > 0 else 0
                day1_mv = day1_pos * day1_price
                
                dollar_pnl = day1_mv - day0_mv
                
                if day0_pos > 0 and day1_pos > 0:
                    inflows_outflows = (day1_pos - day0_pos) * (day1_price + day0_price) / 2
                    adj_pnl = dollar_pnl - inflows_outflows
                elif day0_pos == 0 and day1_pos > 0:
                    inflows = day1_pos * day1_price
                    adj_pnl = dollar_pnl - inflows
                elif day0_pos > 0 and day1_pos == 0:
                    adj_pnl = 0
                else:
                    adj_pnl = 0
                
                pnl_data[ticker] = adj_pnl
            
            positive_pnl = {t: p for t, p in pnl_data.items() if p > 0}
            negative_pnl = {t: p for t, p in pnl_data.items() if p < 0}
            
            profit_count, profit_names = 0, ''
            if positive_pnl:
                sorted_positive = sorted(positive_pnl.items(), key=lambda x: x[1], reverse=True)
                total_positive = sum(p for _, p in sorted_positive)
                
                cumulative, top_tickers = 0, []
                for ticker, pnl in sorted_positive:
                    cumulative += pnl
                    top_tickers.append(ticker)
                    if cumulative >= total_positive * 0.5:
                        break
                
                profit_count = len(top_tickers)
                profit_names = ', '.join(top_tickers)
            
            loss_count, loss_names = 0, ''
            if negative_pnl:
                sorted_negative = sorted(negative_pnl.items(), key=lambda x: x[1])
                total_negative = sum(abs(p) for _, p in sorted_negative)
                
                cumulative, top_tickers = 0, []
                for ticker, pnl in sorted_negative:
                    cumulative += abs(pnl)
                    top_tickers.append(ticker)
                    if cumulative >= total_negative * 0.5:
                        break
                
                loss_count = len(top_tickers)
                loss_names = ', '.join(top_tickers)
            
            contributor_results.append({
                'Date': current_date.strftime('%m/%d/%Y'),
                'Top_50pct_Profit_Count': profit_count,
                'Top_50pct_Profit_Tickers': profit_names,
                'Top_50pct_Loss_Count': loss_count,
                'Top_50pct_Loss_Tickers': loss_names
            })
    
    contributors_df = pd.DataFrame(contributor_results)
    daily_analysis = pd.merge(hhi_dict[etf_name], contributors_df, on='Date', how='left')
    analysis_dict[etf_name] = daily_analysis
    
    # Prepare Excel output with renamed columns
    excel_output = daily_analysis.copy()
    excel_output = excel_output.rename(columns={
        'Holdings_Count': 'Holdings',
        'Top_50pct_Profit_Tickers': '50% Profit Contributors',
        'Top_50pct_Loss_Tickers': '50% Loss Contributors'
    })
    # Drop the count columns
    excel_output = excel_output.drop(columns=['Top_50pct_Profit_Count', 'Top_50pct_Loss_Count'])
    
    # Save to Excel
    output_file = os.path.join(output_dir, f'{etf_name}_Analysis_{datetime.now().strftime("%Y%m%d")}.xlsx')
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        excel_output.to_excel(writer, sheet_name='Daily_HHI_Analysis', index=False)
    
    print(f"  ✅ Saved: {etf_name}_Analysis_{datetime.now().strftime('%Y%m%d')}.xlsx")

print(f"\n✅ P&L calculated and saved for all {len(analysis_dict)} ETFs!")

## Step 4: Generate Visualizations

In [ ]:
for etf_name, daily_analysis in analysis_dict.items():
    print(f"Creating visualizations for {etf_name}...")
    
    # 1. HHI and ENH Time Series
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7))
    plot_data = daily_analysis.copy()
    plot_data['Date'] = pd.to_datetime(plot_data['Date'])

    # HHI subplot
    ax1.plot(plot_data['Date'], plot_data['HHI'], linewidth=2.5, color='navy', alpha=0.8)
    ax1.set_xlabel('Date', fontsize=14, fontweight='bold')
    ax1.set_title(f'{etf_name} Portfolio Concentration (HHI) Over Time', fontsize=16, fontweight='bold', pad=15)
    ax1.set_ylim(0, 0.1)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='both', which='major', labelsize=12)
    ax1.axhline(y=0.01, color='green', linestyle=':', alpha=0.5, linewidth=1.5, label='Highly Diversified (<0.01)')
    ax1.axhline(y=0.15, color='orange', linestyle=':', alpha=0.5, linewidth=1.5, label='Moderately Concentrated (0.15)')
    ax1.axhline(y=0.25, color='red', linestyle=':', alpha=0.5, linewidth=1.5, label='Highly Concentrated (>0.25)')
    ax1.legend(loc='upper right', fontsize=11)
    hhi_mean = plot_data["HHI"].mean()
    hhi_min = plot_data["HHI"].min()
    hhi_max = plot_data["HHI"].max()
    stats_text = f'Mean: {hhi_mean:.4f}\nMin: {hhi_min:.4f}\nMax: {hhi_max:.4f}'
    ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, fontsize=11,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # ENH subplot
    ax2.plot(plot_data['Date'], plot_data['ENH'], linewidth=2.5, color='darkgreen', alpha=0.8)
    ax2.set_xlabel('Date', fontsize=14, fontweight='bold')
    ax2.set_title(f'{etf_name} Effective Number of Holdings (ENH = 1/HHI)', fontsize=16, fontweight='bold', pad=15)
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(axis='both', which='major', labelsize=12)
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
    enh_mean = plot_data["ENH"].mean()
    enh_min = plot_data["ENH"].min()
    enh_max = plot_data["ENH"].max()
    stats_text = f'Mean: {enh_mean:.2f}\nMin: {enh_min:.2f}\nMax: {enh_max:.2f}'
    ax2.text(0.02, 0.98, stats_text, transform=ax2.transAxes, fontsize=11,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{etf_name}_HHI_ENH_TimeSeries.png'), dpi=600, bbox_inches='tight')
    plt.close()
    
    # 2. Frequent Drivers of 50% Daily Profits
    all_tickers = []
    for tickers_str in daily_analysis['Top_50pct_Profit_Tickers']:
        if pd.notna(tickers_str) and tickers_str != '':
            tickers = [t.strip() for t in str(tickers_str).split(',')]
            all_tickers.extend(tickers)
    
    if all_tickers:
        ticker_counts = pd.Series(all_tickers).value_counts()
        top_20 = ticker_counts.head(20)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        bars = ax.bar(range(len(top_20)), top_20.values, color='steelblue', alpha=0.8)
        colors = plt.cm.coolwarm(np.linspace(0.3, 0.7, len(bars)))
        for bar, color in zip(bars, colors):
            bar.set_color(color)
        ax.set_xticks(range(len(top_20)))
        ax.set_xticklabels(top_20.index, rotation=45, ha='right', fontsize=13, fontweight='bold')
        ax.set_xlabel('Stock Ticker', fontsize=15, fontweight='bold')
        ax.set_ylabel('Frequency (Days)', fontsize=15, fontweight='bold')
        ax.set_title(f'{etf_name} - Frequent Drivers of 50% Daily Profits', fontsize=17, fontweight='bold', pad=15)
        ax.tick_params(axis='y', which='major', labelsize=12)
        for bar, value in zip(bars, top_20.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(value), ha='center', va='bottom', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        total_days = (daily_analysis['Top_50pct_Profit_Count'] > 0).sum()
        stats_text = f'Total days with data: {total_days}\nUnique drivers: {len(ticker_counts)}'
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, fontsize=12,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{etf_name}_Frequent_Drivers_50pct_Profits.png'), dpi=600, bbox_inches='tight')
        plt.close()
    
    # 3. Frequent Drivers of 50% Daily Losses
    all_tickers = []
    for tickers_str in daily_analysis['Top_50pct_Loss_Tickers']:
        if pd.notna(tickers_str) and tickers_str != '':
            tickers = [t.strip() for t in str(tickers_str).split(',')]
            all_tickers.extend(tickers)
    
    if all_tickers:
        ticker_counts = pd.Series(all_tickers).value_counts()
        top_20 = ticker_counts.head(20)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        bars = ax.bar(range(len(top_20)), top_20.values, color='indianred', alpha=0.8)
        # Use coolwarm colormap reversed for nice gradient (warm to cool)
        colors = plt.cm.coolwarm(np.linspace(0.7, 0.3, len(bars)))
        for bar, color in zip(bars, colors):
            bar.set_color(color)
        ax.set_xticks(range(len(top_20)))
        ax.set_xticklabels(top_20.index, rotation=45, ha='right', fontsize=13, fontweight='bold')
        ax.set_xlabel('Stock Ticker', fontsize=15, fontweight='bold')
        ax.set_ylabel('Frequency (Days)', fontsize=15, fontweight='bold')
        ax.set_title(f'{etf_name} - Frequent Drivers of 50% Daily Losses', fontsize=17, fontweight='bold', pad=15)
        ax.tick_params(axis='y', which='major', labelsize=12)
        for bar, value in zip(bars, top_20.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(value), ha='center', va='bottom', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        total_days = (daily_analysis['Top_50pct_Loss_Count'] > 0).sum()
        stats_text = f'Total days with loss data: {total_days}\nUnique drivers: {len(ticker_counts)}'
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes, fontsize=12,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{etf_name}_Frequent_Drivers_50pct_Losses.png'), dpi=600, bbox_inches='tight')
        plt.close()
    
    print(f"  ✅ Saved 3 visualizations")

print(f"\n{'='*60}")
print(f"🎉 All {len(analysis_dict)} ETFs processed successfully!")
print(f"{'='*60}")